# Spotify User Behavior — Data Cleaning, Quality Assessment & Statistical Testing

**Purpose:** Load the raw Spotify user-behavior dataset, assess its data quality issues in full, apply documented cleaning decisions, validate the result, and export a dataset for the Excel workbook and Power BI dashboard. The notebook also tests whether the group differences shown on the dashboard are statistically real.

**Author:** Akhil T V  |  **Last updated:** November 2026  |  **Data source:** public Kaggle dataset

**Workflow:** Raw CSV → Quality Assessment (read-only) → Cleaning → Validation → Feature Engineering → Export → Significance Testing → KPI Reconciliation

**Run location:** this notebook lives in `notebooks/`, so paths point to `../data/`.

**Design rule:** nothing is silently deleted or "fixed" without a record. Only exact duplicate rows are removed; every other problem is corrected with a stated reason or marked with a `flag_*` column so downstream analysis can include or exclude it on purpose.

## Step 1 — Setup & Load

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats

pd.set_option('display.max_columns', None)

RAW_PATH = '../data/raw_spotify_dataset.csv'
CLEAN_PATH = '../data/cleaned_spotify_dataset.csv'

df = pd.read_csv(RAW_PATH)
raw_rows = len(df)
print(f"Loaded dataset with {df.shape[0]:,} rows and {df.shape[1]} columns")

# Type conversion only (no values change): signup_date arrives as text.
df['signup_date'] = pd.to_datetime(df['signup_date'])

# The original 17 non-ID columns define a user "profile" (used later for near-duplicate detection).
PROFILE_COLS = [c for c in df.columns if c != 'user_id']

Loaded dataset with 50,050 rows and 18 columns


## Step 2 — Initial Inspection

In [2]:
df.shape

(50050, 18)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50050 entries, 0 to 50049
Data columns (total 18 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   user_id                         50050 non-null  int64         
 1   country                         50050 non-null  object        
 2   age                             48299 non-null  float64       
 3   signup_date                     50050 non-null  datetime64[ns]
 4   subscription_type               50050 non-null  object        
 5   subscription_status             50050 non-null  object        
 6   months_inactive                 50050 non-null  int64         
 7   inactive_3_months_flag          50050 non-null  int64         
 8   ad_interaction                  50050 non-null  object        
 9   ad_conversion_to_subscription   50050 non-null  object        
 10  music_suggestion_rating_1_to_5  45678 non-null  float64       
 11  av

In [4]:
df.head()

,user_id,country,age,signup_date,subscription_type,subscription_status,months_inactive,inactive_3_months_flag,ad_interaction,ad_conversion_to_subscription,music_suggestion_rating_1_to_5,avg_listening_hours_per_week,favorite_genre,most_liked_feature,desired_future_feature,primary_device,playlists_created,avg_skips_per_day
0,17397,USA,38.0,2020-07-18,Free,Inactive,3,1,No,No,5.0,8.37,Jazz,AI DJ,NaN,Mobile,7,7
1,21810,Canada,29.0,2025-01-01,Free,Active,0,0,No,No,3.0,3.82,K-Pop,AI DJ,NaN,Car System,3,12
2,5717,Indonesia,24.0,2023-10-21,Premium Individual,Active,1,0,Yes,No,3.0,17.10,Bollywood,Podcasts,Concert Alerts,Tablet,9,11
3,18847,France,52.0,2025-07-11,Premium Duo,Active,0,0,Yes,No,5.0,6.67,Electronic,Playlists,NaN,Tablet,10,11
4,10745,Spain,36.0,2022-04-27,Free,Active,1,0,No,No,4.0,11.26,R&B,Playlists,Lyrics Translation,Smart Speaker,7,8


In [5]:
df.describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
user_id,50050.0,NaN,NaN,NaN,25006.855584,1.0,12503.25,25004.5,37509.75,50020.0,14438.876837
country,50050,15,India,4269,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,48299.0,NaN,NaN,NaN,37.993892,2.0,27.0,38.0,49.0,120.0,13.035731
signup_date,50050,NaN,NaN,NaN,2022-02-01 20:11:19.576423680,2018-01-01 00:00:00,2020-01-19 00:00:00,2022-02-01 00:00:00,2024-02-16 00:00:00,2027-02-08 00:00:00,NaN
subscription_type,50050,5,Free,22549,NaN,NaN,NaN,NaN,NaN,NaN,NaN
subscription_status,50050,2,Active,42154,NaN,NaN,NaN,NaN,NaN,NaN,NaN
months_inactive,50050.0,NaN,NaN,NaN,1.532547,0.0,0.0,1.0,2.0,18.0,1.951691
inactive_3_months_flag,50050.0,NaN,NaN,NaN,0.222378,0.0,0.0,0.0,0.0,1.0,0.415848
ad_interaction,50050,2,No,32538,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ad_conversion_to_subscription,50050,2,No,45689,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Initial observations:**
- 50,050 rows × 18 columns, one row per `user_id`. Four columns contain nulls, and `desired_future_feature` is 40% empty.
- `describe()` already shows impossible values: `age` runs from 2 to 120, `avg_listening_hours_per_week` from −16.74 to about 162, and `playlists_created` from −1 to 376.
- `country` has 15 distinct values and `primary_device` has 15, although there should be 12 countries and 5 devices. These are spelling and casing variants of the same values.
- Categorical fields such as country, genre and device are spread almost evenly across their categories (see Known Limitations).

## Step 3 — Data Quality Assessment (read-only)

The full assessment is completed **before any change is made**, so every cleaning decision is based on the complete picture. Nothing in this step modifies `df`.

### 3.1 Missing Values

In [6]:
missing_summary = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct': (df.isnull().sum() / len(df) * 100).round(2)
})
missing_summary = missing_summary[missing_summary['missing_count'] > 0].sort_values('missing_count', ascending=False)
missing_summary

,missing_count,missing_pct
desired_future_feature,20021,40.00
music_suggestion_rating_1_to_5,4372,8.74
favorite_genre,2504,5.00
age,1751,3.50


### 3.2 Duplicate Records

In [7]:
n_exact_dupes = int(df.duplicated().sum())
n_dup_ids = int(df['user_id'].duplicated().sum())
print(f"Fully identical duplicate rows: {n_exact_dupes}")
print(f"Duplicate user_id values: {n_dup_ids}")

Fully identical duplicate rows: 30
Duplicate user_id values: 30


In [8]:
# Near-duplicates: identical profile data under a DIFFERENT user_id (a re-signup, or a duplicate/bot account).
# Checked on the data AFTER exact duplicates are set aside, so the 30 exact pairs are not counted twice.
deduped_preview = df.drop_duplicates()
near_mask = deduped_preview.duplicated(subset=PROFILE_COLS, keep=False)
n_near_dupe_rows = int(near_mask.sum())
print(f"Rows sharing an identical profile with another user_id: {n_near_dupe_rows} ({n_near_dupe_rows // 2} pairs)")
deduped_preview[near_mask].sort_values(PROFILE_COLS[:3]).head(6)

Rows sharing an identical profile with another user_id: 40 (20 pairs)


,user_id,country,age,signup_date,subscription_type,subscription_status,months_inactive,inactive_3_months_flag,ad_interaction,ad_conversion_to_subscription,music_suggestion_rating_1_to_5,avg_listening_hours_per_week,favorite_genre,most_liked_feature,desired_future_feature,primary_device,playlists_created,avg_skips_per_day
18521,50006,Australia,59.0,2020-04-27,Free,Active,0,0,Yes,Yes,4.0,9.56,Rock,Discover Weekly,Mood-based Auto Playlists,Tablet,5,8
21722,34672,Australia,59.0,2020-04-27,Free,Active,0,0,Yes,Yes,4.0,9.56,Rock,Discover Weekly,Mood-based Auto Playlists,Tablet,5,8
21925,50014,Canada,42.0,2024-02-05,Premium Individual,Inactive,3,1,Yes,Yes,NaN,4.95,Electronic,Radio,NaN,Smart Speaker,10,8
46135,36837,Canada,42.0,2024-02-05,Premium Individual,Inactive,3,1,Yes,Yes,NaN,4.95,Electronic,Radio,NaN,Smart Speaker,10,8
38105,50004,Canada,44.0,2025-08-17,Free,Active,7,1,Yes,No,4.0,8.54,Latin,Discover Weekly,Social Listening,Tablet,10,12
38925,40377,Canada,44.0,2025-08-17,Free,Active,7,1,Yes,No,4.0,8.54,Latin,Discover Weekly,Social Listening,Tablet,10,12


### 3.3 Invalid / Out-of-Range Values

In [9]:
n_invalid_age = int(((df['age'] < 10) | (df['age'] > 100)).sum())
n_neg_hours = int((df['avg_listening_hours_per_week'] < 0).sum())
n_neg_playlists = int((df['playlists_created'] < 0).sum())
n_extreme_hours = int((df['avg_listening_hours_per_week'] > 100).sum())
n_extreme_playlists = int((df['playlists_created'] > 150).sum())

print("Age range:", df['age'].min(), "to", df['age'].max())
print("Implausible age (<10 or >100):", n_invalid_age)
print()
print("Negative avg_listening_hours_per_week:", n_neg_hours)
print("Negative playlists_created:", n_neg_playlists)
print()
print("Extreme avg_listening_hours_per_week (>100 hrs/week):", n_extreme_hours)
print("Extreme playlists_created (>150):", n_extreme_playlists)

Age range: 2.0 to 120.0
Implausible age (<10 or >100): 25

Negative avg_listening_hours_per_week: 15
Negative playlists_created: 10

Extreme avg_listening_hours_per_week (>100 hrs/week): 12
Extreme playlists_created (>150): 10


### 3.4 Categorical Text Consistency

In [10]:
print("Unique country values:")
print(sorted(df['country'].unique()))
print()
print("Unique primary_device values:")
print(sorted(df['primary_device'].unique()))

uk_rows = df['country'].str.strip().str.lower().isin(['uk', 'u.k.', 'united kingdom'])
n_uk_rows = int(uk_rows.sum())
n_uk_nonstandard = int((uk_rows & (df['country'] != 'United Kingdom')).sum())
n_device_nonstandard = int((df['primary_device'] != df['primary_device'].str.strip().str.title()).sum())
print()
print(f"UK appears under 4 spellings across {n_uk_rows:,} rows ({n_uk_nonstandard:,} differ from 'United Kingdom')")
print(f"primary_device rows with non-standard casing: {n_device_nonstandard:,}")

Unique country values:
['Australia', 'Brazil', 'Canada', 'France', 'Germany', 'India', 'Indonesia', 'Italy', 'Mexico', 'Spain', 'U.K.', 'UK', 'USA', 'United Kingdom', 'uk']

Unique primary_device values:
['CAR SYSTEM', 'Car System', 'DESKTOP', 'Desktop', 'MOBILE', 'Mobile', 'SMART SPEAKER', 'Smart Speaker', 'TABLET', 'Tablet', 'car system', 'desktop', 'mobile', 'smart speaker', 'tablet']

UK appears under 4 spellings across 4,149 rows (3,099 differ from 'United Kingdom')
primary_device rows with non-standard casing: 2,643


### 3.5 Timeline Consistency

There is no external "today" for this dataset, so the valid signup window is read from the data itself: find where continuous daily signups stop.

In [11]:
daily = df.groupby('signup_date').size()

# Signups arrive continuously, a few to dozens every day, and then stop abruptly.
LAST_DENSE_DATE = daily[daily >= 3].index.max()
before = daily[daily.index <= LAST_DENSE_DATE]
after = daily[daily.index > LAST_DENSE_DATE]

print("Last date of continuous signups:", LAST_DENSE_DATE.date())
print(f"Days up to that date: {len(before):,}, minimum signups on any day: {before.min()}")
print(f"Signups dated after it: {int(after.sum())} rows on {len(after)} isolated days (maximum {after.max()} per day)")
print("Latest signup date in file:", df['signup_date'].max().date())

n_out_of_window = int((df['signup_date'] > LAST_DENSE_DATE).sum())

Last date of continuous signups: 2026-03-01
Days up to that date: 2,982, minimum signups on any day: 3
Signups dated after it: 20 rows on 20 isolated days (maximum 1 per day)
Latest signup date in file: 2027-02-08


### 3.6 Logical Consistency Between Fields

In [12]:
is_free = df['subscription_type'] == 'Free'
conv_yes = df['ad_conversion_to_subscription'] == 'Yes'

n_free_conv = int((is_free & conv_yes).sum())
n_paid_conv = int((~is_free & conv_yes).sum())
n_active_flagged = int(((df['subscription_status'] == 'Active') & (df['inactive_3_months_flag'] == 1)).sum())
n_zero_hours = int((df['avg_listening_hours_per_week'] == 0).sum())
flag_matches_months = ((df['months_inactive'] >= 3).astype(int) == df['inactive_3_months_flag']).all()

print(f"Ad conversion = Yes but still on the Free plan: {n_free_conv}")
print(f"Ad conversion = Yes on a paid plan (consistent with the label): {n_paid_conv}")
print(f"Status 'Active' but flagged inactive for 3+ months: {n_active_flagged}")
print(f"Users with exactly 0 listening hours: {n_zero_hours}")
print(f"inactive_3_months_flag matches months_inactive >= 3 everywhere: {flag_matches_months}")
print(f"Conversion = Yes without any ad interaction: {int(((df['ad_interaction']=='No') & conv_yes).sum())}")

Ad conversion = Yes but still on the Free plan: 1925
Ad conversion = Yes on a paid plan (consistent with the label): 2436
Status 'Active' but flagged inactive for 3+ months: 3234
Users with exactly 0 listening hours: 327
inactive_3_months_flag matches months_inactive >= 3 everywhere: True
Conversion = Yes without any ad interaction: 0


### Data Quality Assessment Summary (counts computed from the raw data)

In [13]:
quality_summary = pd.DataFrame([
    ('age',                           'Missing',                                           int(df['age'].isna().sum()),                     'Medium'),
    ('age',                           'Implausible (<10 or >100)',                         n_invalid_age,                                   'High'),
    ('favorite_genre',                'Missing',                                           int(df['favorite_genre'].isna().sum()),          'Low'),
    ('music_suggestion_rating_1_to_5','Missing',                                           int(df['music_suggestion_rating_1_to_5'].isna().sum()), 'Medium'),
    ('desired_future_feature',        'Missing (optional field)',                          int(df['desired_future_feature'].isna().sum()),  'Low'),
    ('country',                       'UK under 4 spellings (rows differing from standard)', n_uk_nonstandard,                              'Medium'),
    ('primary_device',                'Inconsistent casing',                               n_device_nonstandard,                            'Low'),
    ('avg_listening_hours_per_week',  'Negative values',                                   n_neg_hours,                                     'High'),
    ('playlists_created',             'Negative values',                                   n_neg_playlists,                                 'High'),
    ('(all columns)',                 'Exact duplicate rows',                              n_exact_dupes,                                   'High'),
    ('(all columns)',                 'Near-duplicate rows (identical profile, different id)', n_near_dupe_rows,                            'Medium'),
    ('signup_date',                   'Outside the continuous signup window',              n_out_of_window,                                 'Medium'),
    ('listening hours / playlists',   'Extreme but possible values (flag only)',           n_extreme_hours + n_extreme_playlists,           'Low'),
    ('ad_conversion vs subscription_type', 'Conversion = Yes while on Free plan',          n_free_conv,                                     'Medium'),
    ('subscription_status vs inactive flag', "'Active' but inactive 3+ months",            n_active_flagged,                                'Medium'),
    ('avg_listening_hours_per_week',  'Exactly zero hours',                                n_zero_hours,                                    'Low'),
], columns=['Column', 'Issue', 'Rows', 'Severity'])
quality_summary

,Column,Issue,Rows,Severity
0,age,Missing,1751,Medium
1,age,Implausible (<10 or >100),25,High
2,favorite_genre,Missing,2504,Low
3,music_suggestion_rating_1_to_5,Missing,4372,Medium
4,desired_future_feature,Missing (optional field),20021,Low
5,country,UK under 4 spellings (rows differing from stan...,3099,Medium
6,primary_device,Inconsistent casing,2643,Low
7,avg_listening_hours_per_week,Negative values,15,High
8,playlists_created,Negative values,10,High
9,(all columns),Exact duplicate rows,30,High


## Step 4 — Remove Exact Duplicate Rows

**Decision:** Exact duplicate rows are removed. They are the same user record exported twice (identical `user_id` and identical values), so they add no information and would double-count that user in every KPI.

In [14]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(df)} exact duplicate rows. New row count: {len(df):,}")
print("user_id is now unique:", df['user_id'].is_unique)

Removed 30 exact duplicate rows. New row count: 50,020
user_id is now unique: True


## Step 5 — Handle Invalid / Impossible Values

Invalid values are handled **before** missing values, so the missing-value step sees the final set of nulls.

**Age:** values under 10 or over 100 are data-entry errors. They are set to null rather than guessed or dropped, and marked in `flag_age_invalid`.

**Negative listening hours / playlists:** a negative duration or count is impossible. The correction takes the absolute value, on the assumption of an upstream sign error: the magnitudes (for example 16.74 hours) sit inside the normal range once the sign is flipped. This is an assumption, not a certainty, so every corrected row is marked in `flag_negative_corrected` and can be excluded. It affects 25 rows, about 0.05% of the data.

**Extreme but possible values:** more than 100 hours/week or more than 150 playlists is unusual but not impossible. These rows are flagged, not deleted or capped, so genuine power users are not erased.

In [15]:
# --- Age: flag, then null
df['flag_age_invalid'] = (df['age'] < 10) | (df['age'] > 100)
print(f"Invalid age values set to null: {df['flag_age_invalid'].sum()}")
df.loc[df['flag_age_invalid'], 'age'] = np.nan

Invalid age values set to null: 25


In [16]:
# --- Negative values: flag, then absolute value
df['flag_negative_corrected'] = (df['avg_listening_hours_per_week'] < 0) | (df['playlists_created'] < 0)
print(f"Negative listening hours corrected: {(df['avg_listening_hours_per_week'] < 0).sum()}")
print(f"Negative playlist counts corrected: {(df['playlists_created'] < 0).sum()}")
df['avg_listening_hours_per_week'] = df['avg_listening_hours_per_week'].abs()
df['playlists_created'] = df['playlists_created'].abs()

Negative listening hours corrected: 15
Negative playlist counts corrected: 10


In [17]:
# --- Extreme outliers: flag only
df['flag_outlier_listening_hours'] = df['avg_listening_hours_per_week'] > 100
df['flag_outlier_playlists'] = df['playlists_created'] > 150
print(f"Outlier listening hours flagged: {df['flag_outlier_listening_hours'].sum()}")
print(f"Outlier playlist counts flagged: {df['flag_outlier_playlists'].sum()}")

Outlier listening hours flagged: 12
Outlier playlist counts flagged: 10


## Step 6 — Handle Missing Values

Each column is handled on its own. One blanket rule ("fill with the mean" or "drop every row with a null") is wrong here, because the right treatment depends on what the column means and why it is empty.

### 6.1 `age`
**Decision:** leave as null; do not impute.

**Justification:** no other column is a reliable proxy for age, and filling with a mean or median would invent a fact about each user and distort age-group analysis. Rows with unknown age are simply left out of age breakdowns. After Step 5 the nulls are the originally missing ages plus the invalid ages set to null.

In [18]:
print(f"Rows with null age (missing + invalid): {df['age'].isnull().sum():,}")

Rows with null age (missing + invalid): 1,775


### 6.2 `favorite_genre`
**Decision:** fill with `'Not Specified'`.

**Justification:** this is a categorical preference field. The user never set a preference, which is meaningful in itself, so it becomes its own category and is not dropped or guessed. Statistical tests later exclude this placeholder because it is not a genre.

In [19]:
df['favorite_genre'] = df['favorite_genre'].fillna('Not Specified')
print(df['favorite_genre'].isnull().sum(), "missing values remaining")

0 missing values remaining


### 6.3 `music_suggestion_rating_1_to_5`
**Decision:** leave as null; do not impute. Add `flag_no_rating_given` so "never rated" can be analysed as its own segment.

**Justification:** the missing ratings are not random. The check below shows that users with no rating listen only about half as much as users who rated. Filling the gaps with an average rating would wrongly make disengaged users look satisfied.

In [20]:
df['flag_no_rating_given'] = df['music_suggestion_rating_1_to_5'].isnull()

# Evidence that "no rating" is linked to low engagement
print(df.groupby('flag_no_rating_given')['avg_listening_hours_per_week'].agg(['count', 'mean']).round(2)
        .rename(index={False: 'rated', True: 'no rating'}))

                      count   mean
flag_no_rating_given              
rated                 45649  10.50
no rating              4371   4.97


### 6.4 `desired_future_feature`
**Decision:** fill with `'No Response'`.

**Justification:** this is an optional open-feedback field, so a high empty rate (40%) is expected. A clear placeholder keeps the column usable in counts without implying an opinion the user never gave.

In [21]:
df['desired_future_feature'] = df['desired_future_feature'].fillna('No Response')
print(df['desired_future_feature'].isnull().sum(), "missing values remaining")

0 missing values remaining


## Step 7 — Standardize Categorical Text

**Decision:** map every spelling and casing variant of the same country to one standard name, and put device names in consistent title case.

**Justification:** without this, grouping would treat `UK`, `uk`, `U.K.` and `United Kingdom` as four countries, splitting one country's users across four bars and understating each. `USA` is renamed to `United States` so country names are full and consistent.

In [22]:
country_mapping = {
    'usa': 'United States', 'u.s.a.': 'United States', 'u.s.a': 'United States', 'us': 'United States',
    'united states': 'United States', 'united states of america': 'United States',
    'uk': 'United Kingdom', 'u.k.': 'United Kingdom', 'u.k': 'United Kingdom',
    'united kingdom': 'United Kingdom', 'great britain': 'United Kingdom',
}
clean_country = df['country'].str.strip()
df['country'] = clean_country.str.lower().map(country_mapping).fillna(clean_country)
print(sorted(df['country'].unique()))

['Australia', 'Brazil', 'Canada', 'France', 'Germany', 'India', 'Indonesia', 'Italy', 'Mexico', 'Spain', 'United Kingdom', 'United States']


In [23]:
df['primary_device'] = df['primary_device'].str.strip().str.title()
print(sorted(df['primary_device'].unique()))

['Car System', 'Desktop', 'Mobile', 'Smart Speaker', 'Tablet']


## Step 8 — Flag Near-Duplicate Profiles

**Decision:** flag, do not delete.

**Justification:** two accounts with an identical profile but different `user_id` could be a legitimate re-signup after account deletion, or a duplicate/bot account. The data cannot say which, so the rows are kept and marked in `flag_duplicate_profile` for a stakeholder to review. This runs **after** standardization so that records which differ only by spelling or casing (for example `UK` vs `United Kingdom`) are compared correctly.

In [24]:
df['flag_duplicate_profile'] = df.duplicated(subset=PROFILE_COLS, keep=False)
print(f"Flagged {df['flag_duplicate_profile'].sum()} rows ({df['flag_duplicate_profile'].sum() // 2} pairs) as duplicate profiles for review")

Flagged 40 rows (20 pairs) as duplicate profiles for review


## Step 9 — Flag Signup Dates Outside the Observed Window

**Decision:** flag, do not correct. Rows are kept, and their Year and Month are left empty in Step 12 so trend charts exclude them automatically.

**Justification:** Step 3.5 showed that signups are continuous until `LAST_DENSE_DATE` and then appear only as isolated single rows scattered across the following year. There is no way to know the true dates, so inventing replacements would add false precision. Flagging and excluding them from time trends is the honest option.

The column keeps the name `flag_invalid_signup_date` for compatibility with the dashboard. The cutoff is read from the data in Step 3.5, not typed in by hand.

In [25]:
df['flag_invalid_signup_date'] = df['signup_date'] > LAST_DENSE_DATE
print(f"Cutoff (last day of continuous signups): {LAST_DENSE_DATE.date()}")
print(f"Rows flagged: {df['flag_invalid_signup_date'].sum()}")

Cutoff (last day of continuous signups): 2026-03-01
Rows flagged: 20


## Step 10 — Flag Logical Conflicts Between Fields

**Decision:** flag, do not alter. These fields come from the source system and cannot be fixed without business input, but they change how the KPIs must be defined.

- **`flag_conversion_plan_mismatch`**: `ad_conversion_to_subscription = Yes` while `subscription_type = Free`. A user who converted to a subscription should not still be on the Free plan (they may have converted and later downgraded, but the data has no event history). Conversions on paid plans are consistent with the label and are *not* flagged.
- **`flag_active_but_inactive_3m`**: `subscription_status = Active` but `inactive_3_months_flag = 1`. The two fields measure different things (account status vs. recent activity), so a user can be "Active" on paper while not listening for 3+ months. This is why the project uses two separately named churn measures (see Step 15).
- **`flag_zero_listening`**: exactly 0 hours/week. These are plausibly genuine dormant users, so they are kept and marked.

In [26]:
df['flag_conversion_plan_mismatch'] = (df['subscription_type'] == 'Free') & (df['ad_conversion_to_subscription'] == 'Yes')
df['flag_active_but_inactive_3m'] = (df['subscription_status'] == 'Active') & (df['inactive_3_months_flag'] == 1)
df['flag_zero_listening'] = df['avg_listening_hours_per_week'] == 0

for c in ['flag_conversion_plan_mismatch', 'flag_active_but_inactive_3m', 'flag_zero_listening']:
    print(f"{c}: {df[c].sum():,} rows")

flag_conversion_plan_mismatch: 1,924 rows
flag_active_but_inactive_3m: 3,233 rows
flag_zero_listening: 327 rows


## Step 11 — Final Validation

The Step 3 checks are re-run on the cleaned data, with assertions so that any regression stops the notebook instead of passing unnoticed.

In [27]:
print("=== FINAL VALIDATION ===\n")

print("Remaining missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print()

checks = {
    "No exact duplicate rows":              df.duplicated().sum() == 0,
    "user_id is unique":                    df['user_id'].is_unique,
    "No negative listening hours":          (df['avg_listening_hours_per_week'] < 0).sum() == 0,
    "No negative playlist counts":          (df['playlists_created'] < 0).sum() == 0,
    "All ages null or within 10-100":       df['age'].dropna().between(10, 100).all(),
    "12 standardized countries":            df['country'].nunique() == 12,
    "5 standardized devices":               df['primary_device'].nunique() == 5,
    "Only intended columns have nulls":     set(df.columns[df.isnull().any()]) == {'age', 'music_suggestion_rating_1_to_5'},
    "Rows removed = exact duplicates only": raw_rows - len(df) == n_exact_dupes,
}
for name, ok in checks.items():
    print(("PASS  " if ok else "FAIL  ") + name)
assert all(checks.values()), "Validation failed - see FAIL lines above"

print()
print(f"Valid age range after cleaning: {df['age'].min():.0f} to {df['age'].max():.0f}")
print(f"Final dataset shape before feature engineering: {df.shape}")

=== FINAL VALIDATION ===

Remaining missing values:
age                               1775
music_suggestion_rating_1_to_5    4371
dtype: int64

PASS  No exact duplicate rows
PASS  user_id is unique
PASS  No negative listening hours
PASS  No negative playlist counts
PASS  All ages null or within 10-100
PASS  12 standardized countries
PASS  5 standardized devices
PASS  Only intended columns have nulls
PASS  Rows removed = exact duplicates only

Valid age range after cleaning: 16 to 60
Final dataset shape before feature engineering: (50020, 28)


**Note on remaining nulls:** `age` and `music_suggestion_rating_1_to_5` contain nulls by design (Steps 6.1 and 6.3). Every other field is complete.

## Step 12 — Feature Engineering: Helper Columns for Analysis

These derived columns use the same bucketing logic as the Excel workbook, so segment definitions stay identical across the notebook, Excel and Power BI. The bins were confirmed against the original workbook, not re-guessed.

In [28]:
# Age Group (bins are right-inclusive: Gen Z = 16-24, ..., Seniors = 55-61)
age_bins = [15, 24, 34, 44, 54, 61]
age_labels = ['Gen Z', 'Young Adults', 'Mid-Career', 'Mature Adults', 'Seniors']
df['Age Group'] = pd.cut(df['age'], bins=age_bins, labels=age_labels)
print(df['Age Group'].value_counts(dropna=False))

Age Group
Mature Adults    10893
Mid-Career       10705
Young Adults     10618
Gen Z             9688
Seniors           6341
NaN               1775
Name: count, dtype: int64


In [29]:
# Signup Month / Year. Left empty for out-of-window signups (Step 9) so time trends exclude them.
in_window = ~df['flag_invalid_signup_date']
df['Month'] = df['signup_date'].dt.month.where(in_window).astype('Int64')
df['Year'] = df['signup_date'].dt.year.where(in_window).astype('Int64')
print(df[['signup_date', 'Month', 'Year']].head())
print("Rows with empty Year/Month:", df['Year'].isna().sum())

  signup_date  Month  Year
0  2020-07-18      7  2020
1  2025-01-01      1  2025
2  2023-10-21     10  2023
3  2025-07-11      7  2025
4  2022-04-27      4  2022
Rows with empty Year/Month: 20


In [30]:
# Listening Hour Category: Low <=5 | Medium 5-15 | High 15-24.64 | Very High above 24.64
def listening_category(hrs):
    if pd.isna(hrs):
        return np.nan
    elif hrs <= 5:
        return 'Low'
    elif hrs <= 15:
        return 'Medium'
    elif hrs <= 24.64:
        return 'High'
    else:
        return 'Very High'

df['Listening Hour Category'] = df['avg_listening_hours_per_week'].apply(listening_category)
print(df['Listening Hour Category'].value_counts(dropna=False))

Listening Hour Category
Medium       39561
Low           5345
High          5095
Very High       19
Name: count, dtype: int64


In [31]:
# Playlist Creator Category: None 0 | Casual 1-5 | Active 6-15 | Power User 16+
# Values above 23 are the flagged extreme outliers; they fall into 'Power User' because the
# workbook's top bin has no upper limit.
def playlist_category(n):
    if pd.isna(n):
        return np.nan
    elif n == 0:
        return 'None'
    elif n <= 5:
        return 'Casual'
    elif n <= 15:
        return 'Active'
    else:
        return 'Power User'

df['Playlist Creator Category'] = df['playlists_created'].apply(playlist_category)
print(df['Playlist Creator Category'].value_counts(dropna=False))

Playlist Creator Category
Active        39947
Casual         9628
Power User      432
None             13
Name: count, dtype: int64


In [32]:
# Premium flag: any subscription_type other than 'Free'
df['is_premium'] = df['subscription_type'] != 'Free'
print(df['is_premium'].value_counts())

is_premium
True     27484
False    22536
Name: count, dtype: int64


In [33]:
# Churn flag (account-level): subscription_status == 'Inactive'.
# NOT the same as inactive_3_months_flag - see Step 15 for the two definitions.
df['is_churned'] = (df['subscription_status'] == 'Inactive').astype(int)
print(df['is_churned'].value_counts())

is_churned
0    42126
1     7894
Name: count, dtype: int64


## Step 13 — Export Cleaned Dataset

In [34]:
df.to_csv(CLEAN_PATH, index=False)
print(f"Exported {df.shape[0]:,} rows x {df.shape[1]} columns to {CLEAN_PATH}")

Exported 50,020 rows x 35 columns to ../data/cleaned_spotify_dataset.csv


## Step 14 — Statistical Significance Testing

Several dashboard charts show differences between groups (for example churn by plan, rating by genre). Before treating any of them as a finding, test whether the difference is statistically real or within normal random variation.

- **Chi-square test of independence:** for categorical outcomes across groups (churn across plans or countries).
- **One-way ANOVA:** for a numeric average across more than two groups (rating by genre, listening hours by age group).
- **Effect size:** with about 50,000 users even tiny differences can reach significance, so each test also reports an effect size: Cramér's V for chi-square and eta-squared for ANOVA. By convention, Cramér's V below 0.1 and eta-squared below 0.01 are both "small" effects.
- A p-value below 0.05 is the significance threshold.

Because the project has two churn measures (Step 15), churn is tested under **both**. Rating-by-genre excludes the `Not Specified` placeholder, and listening hours exclude the 12 flagged outliers (>100 hrs/week), which would otherwise dominate an ANOVA.

In [35]:
def cramers_v(ct):
    chi2 = stats.chi2_contingency(ct, correction=False)[0]
    return float(np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1))))

def eta_squared(groups):
    allv = np.concatenate(groups)
    grand = allv.mean()
    ss_between = sum(len(g) * (g.mean() - grand) ** 2 for g in groups)
    return float(ss_between / ((allv - grand) ** 2).sum())

results = []

# Chi-square tests: churn under both definitions, by plan and by country
for outcome_label, outcome_col in [('account status Inactive', 'subscription_status'),
                                   ('inactive 3+ months', 'inactive_3_months_flag')]:
    for group_label, group_col in [('subscription_type', 'subscription_type'), ('country', 'country')]:
        ct = pd.crosstab(df[group_col], df[outcome_col])
        chi2, p, dof, expected = stats.chi2_contingency(ct)
        results.append((f"Churn ({outcome_label}) by {group_label}", 'Chi-square', p, cramers_v(ct)))

# ANOVA: rating by genre (real genres only)
rated = df[(df['favorite_genre'] != 'Not Specified')]
g = [x['music_suggestion_rating_1_to_5'].dropna().values for _, x in rated.groupby('favorite_genre')]
results.append(('Avg recommendation rating by genre', 'ANOVA', stats.f_oneway(*g).pvalue, eta_squared(g)))

# ANOVA: listening hours by age group (outliers excluded)
normal = df[~df['flag_outlier_listening_hours']]
g = [x['avg_listening_hours_per_week'].values for _, x in normal.groupby('Age Group', observed=True)]
results.append(('Avg listening hours by Age Group', 'ANOVA', stats.f_oneway(*g).pvalue, eta_squared(g)))

In [36]:
sig_results = pd.DataFrame(results, columns=['Comparison', 'Test Used', 'p_value', 'Effect size'])
sig_results['Significant (p < 0.05)?'] = sig_results['p_value'] < 0.05
sig_results['Verdict'] = sig_results['Significant (p < 0.05)?'].map({
    True: 'Real pattern - safe to present as a finding',
    False: 'Not significant - should NOT be presented as a real difference'
})
sig_results['p_value'] = sig_results['p_value'].round(4)
sig_results['Effect size'] = sig_results['Effect size'].round(5)
sig_results

,Comparison,Test Used,p_value,Effect size,Significant (p < 0.05)?,Verdict
0,Churn (account status Inactive) by subscriptio...,Chi-square,0.5591,0.00773,False,Not significant - should NOT be presented as a...
1,Churn (account status Inactive) by country,Chi-square,0.5065,0.01433,False,Not significant - should NOT be presented as a...
2,Churn (inactive 3+ months) by subscription_type,Chi-square,0.5821,0.00756,False,Not significant - should NOT be presented as a...
3,Churn (inactive 3+ months) by country,Chi-square,0.2400,0.01665,False,Not significant - should NOT be presented as a...
4,Avg recommendation rating by genre,ANOVA,0.9081,0.00013,False,Not significant - should NOT be presented as a...
5,Avg listening hours by Age Group,ANOVA,0.2601,0.00011,False,Not significant - should NOT be presented as a...


### Interpretation

**None of the six comparisons is statistically significant.** Every p-value is above 0.05, and the smallest is 0.24.

The effect sizes show that this is not just a matter of too little data. With about 50,000 users the tests can detect very small differences, yet the largest effect size is Cramér's V = 0.017 (a "small" effect starts at 0.1), and both ANOVA effect sizes are about 0.0001 (a "small" effect starts at 0.01). The groups are, for practical purposes, the same.

The result holds under **both** churn definitions (account status Inactive and inactive 3+ months). Under the account-status definition, churn differs by less than one percentage point between plans and by about two points between countries, which is the size of difference that random variation alone produces in groups this big.

**What this means for the dashboard:** the charts that compare churn by subscription type or country, recommendation rating by genre, and listening hours by age group are descriptive breakdowns only. Their captions should say the differences are not statistically significant, and no insight or recommendation should claim that one plan, country, genre or age group performs better than another on these metrics.


## Step 15 — Headline KPIs and Metric Definitions

Several KPIs can be computed in more than one defensible way. This table states each definition explicitly, so every dashboard card can be traced to exactly one of them.

In [37]:
n = len(df)
hrs = df['avg_listening_hours_per_week']
ad_users = df['ad_interaction'] == 'Yes'
conv = df['ad_conversion_to_subscription'] == 'Yes'

kpis = pd.DataFrame([
    ('Total users (cleaned)',                                    f"{n:,}",                                                'Count of unique user_id after removing exact duplicates'),
    ('Churn - account status Inactive (is_churned)',             f"{df['is_churned'].mean():.2%}",                       "subscription_status == 'Inactive'"),
    ('Churn - inactive 3+ months (inactive_3_months_flag)',      f"{df['inactive_3_months_flag'].mean():.2%}",           'months_inactive >= 3'),
    ('Premium adoption',                                         f"{df['is_premium'].mean():.2%}",                       "subscription_type != 'Free' (includes Student)"),
    ('Ad interaction rate',                                      f"{ad_users.mean():.2%}",                               "ad_interaction == 'Yes', all users"),
    ('Ad conversion - all users',                                f"{conv.mean():.2%}",                                   "ad_conversion == 'Yes' / all users (includes Free-plan mismatches)"),
    ('Ad conversion - confirmed on a paid plan',                 f"{(conv & df['is_premium']).sum() / n:.2%}",           "conversion Yes AND paid plan / all users"),
    ('Ad conversion - among users who interacted with ads',      f"{(conv & ad_users).sum() / ad_users.sum():.2%}",      'conversion Yes / users with ad_interaction Yes'),
    ('Avg weekly listening hours - all users',                   f"{hrs.mean():.2f}",                                    'Mean of avg_listening_hours_per_week'),
    ('Avg weekly listening hours - excluding 12 outliers',       f"{hrs[~df['flag_outlier_listening_hours']].mean():.2f}", 'Same, excluding rows flagged >100 hrs/week'),
], columns=['KPI', 'Value', 'Definition'])
kpis

,KPI,Value,Definition
0,Total users (cleaned),"50,020",Count of unique user_id after removing exact d...
1,Churn - account status Inactive (is_churned),15.78%,subscription_status == 'Inactive'
2,Churn - inactive 3+ months (inactive_3_months_...,22.25%,months_inactive >= 3
3,Premium adoption,54.95%,subscription_type != 'Free' (includes Student)
4,Ad interaction rate,34.98%,"ad_interaction == 'Yes', all users"
5,Ad conversion - all users,8.71%,ad_conversion == 'Yes' / all users (includes F...
6,Ad conversion - confirmed on a paid plan,4.87%,conversion Yes AND paid plan / all users
7,Ad conversion - among users who interacted wit...,24.91%,conversion Yes / users with ad_interaction Yes
8,Avg weekly listening hours - all users,10.02,Mean of avg_listening_hours_per_week
9,Avg weekly listening hours - excluding 12 outl...,9.99,"Same, excluding rows flagged >100 hrs/week"


**Use one definition per dashboard card and label it.** In particular, "churn" must say whether it means account status or 3-month inactivity, and "ad conversion" must say which base it divides by. The Chi-square results above hold under both churn definitions.

## Summary

In [38]:
flag_cols = [c for c in df.columns if c.startswith('flag_')]
print(f"Starting row count:  {raw_rows:,}")
print(f"Final row count:     {len(df):,}")
print(f"Rows removed:        {raw_rows - len(df)} (exact duplicates only)")
print(f"Final columns:       {df.shape[1]} (18 original + {len(flag_cols)} flag columns + derived helper columns)")
print(f"Intentional nulls:   age ({df['age'].isna().sum():,}), music_suggestion_rating_1_to_5 ({df['music_suggestion_rating_1_to_5'].isna().sum():,})")
print()
print("Flag columns (rows flagged):")
for c in flag_cols:
    print(f"  {c:34s} {int(df[c].sum()):>6,}")

Starting row count:  50,050
Final row count:     50,020
Rows removed:        30 (exact duplicates only)
Final columns:       35 (18 original + 10 flag columns + derived helper columns)
Intentional nulls:   age (1,775), music_suggestion_rating_1_to_5 (4,371)

Flag columns (rows flagged):
  flag_age_invalid                       25
  flag_negative_corrected                25
  flag_outlier_listening_hours           12
  flag_outlier_playlists                 10
  flag_no_rating_given                4,371
  flag_duplicate_profile                 40
  flag_invalid_signup_date               20
  flag_conversion_plan_mismatch       1,924
  flag_active_but_inactive_3m         3,233
  flag_zero_listening                   327


### Known limitations
- **Near-duplicate profiles were flagged, not resolved.** Whether they are legitimate re-signups or duplicate/bot accounts needs business input.
- **Negative values were corrected with `abs()`** on the assumption of a sign error. The 25 affected rows are marked in `flag_negative_corrected` and can be excluded.
- **Field conflicts were flagged, not repaired** (Step 10): the source data has no event history to show what really happened.
- **The signup window is inferred from the data** (Step 3.5), not from documentation.
- **The data looks simulated.** Country, genre, device and feature categories are almost perfectly evenly distributed, and valid ages run only from 16 to 60. Findings describe *this dataset* and must not be presented as real-world Spotify user behavior.
- **No causal claims.** The data is a single snapshot with no event timeline, so it supports description and association only.